# Frontier-Tier LLM Feature Generation
### Add 4 frontier LLMs (GPT-5.5, Claude Opus 4.6, Gemini 3.1 Pro, DeepSeek V4 Pro) to the study by **appending** to existing `llm_suggestions.json` and `llm_metrics.csv` from notebook 03.

##### Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import json
import time

import pandas as pd

from src.config import  RESULTS_DIR
from src.data_loader import load_churn, load_housing, load_bank
from src.llm_client import LLMClient
from src.prompts import (
    DATASET_CONFIG,
    build_zero_shot_prompt,
    build_with_stats_prompt,
)
from src.feature_engineer import apply_features

### Load datasets and existing production-tier suggestions
##### Reload `llm_suggestions.json` from notebook 03 (36 entries). We'll append the 24 new frontier entries to this dict in-memory and re-save at the end.

In [ ]:
datasets = {
    "churn": load_churn(verbose=False),
    "housing": load_housing(verbose=False),
    "bank": load_bank(verbose=False),
}

# Load existing suggestions from the previous 6-LLM run
with open(RESULTS_DIR / "llm_suggestions.json") as f:
    all_suggestions = json.load(f)

total_existing = sum(len(v) for v in all_suggestions.values())
print(f"Existing suggestions: {total_existing} (LLM × prompt) entries")

### Build jobs — frontier LLMs only
##### 3 datasets × 4 frontier LLMs × 2 prompt variants = **24 new jobs**. Production-tier LLMs are skipped here since their suggestions are already saved.

In [ ]:
from config import FRONTIER_LLMS
from itertools import product

prompt_variants = ["zero_shot", "with_stats"]

jobs = []
for ds_name, variant, llm_key in product(
    datasets.keys(), prompt_variants, FRONTIER_LLMS
):
    X, _ = datasets[ds_name]
    cfg = DATASET_CONFIG[ds_name]

    if variant == "zero_shot":
        prompt = build_zero_shot_prompt(
            dataset_name=cfg["display_name"],
            task=cfg["task"],
            target=cfg["target"],
            columns=list(X.columns),
        )
    else:
        prompt = build_with_stats_prompt(
            dataset_name=cfg["display_name"],
            task=cfg["task"],
            target=cfg["target"],
            df=X,
        )

    jobs.append(
        {
            "dataset": ds_name,
            "llm": llm_key,
            "variant": variant,
            "prompt": prompt,
        }
    )

print(f"{len(jobs)} new jobs queued (frontier models only)")

### Run the 24 frontier-LLM calls
##### Same loop pattern as notebook 03, but only for the 4 new LLMs.

In [ ]:
new_metrics = []
errors = []
start = time.perf_counter()

for i, job in enumerate(jobs, 1):
    key = f"{job['llm']}__{job['variant']}"
    print(
        f"[{i:2d}/{len(jobs)}] {job['dataset']:8s} | {job['llm']:16s} | {job['variant']:10s}",
        end=" ",
    )

    try:
        llm = LLMClient(job["llm"])
        features, metrics = llm.suggest_features(
            prompt=job["prompt"],
            dataset_name=job["dataset"],
            prompt_variant=job["variant"],
        )
        all_suggestions[job["dataset"]][key] = features

        new_metrics.append(
            {
                "dataset": job["dataset"],
                "llm": job["llm"],
                "variant": job["variant"],
                "n_features": len(features),
                **metrics.to_dict(),
            }
        )

        print(
            f"→ {len(features)} feats | {metrics.total_tokens:>5} tok | ${metrics.cost_usd:.4f} | {metrics.latency_s:.1f}s ✓"
        )
    except Exception as e:
        errors.append({**job, "error": str(e)})
        all_suggestions[job["dataset"]][key] = []
        print(f"→ ERROR: {type(e).__name__}: {str(e)[:80]} ✗")

elapsed = time.perf_counter() - start
total_cost = sum(m["cost_usd"] for m in new_metrics)
print(f"\nTotal time:  {elapsed:.1f}s ({elapsed/60:.1f} min)")
print(f"New cost:    ${total_cost:.4f}")
print(f"Errors:      {len(errors)}/{len(jobs)}")

### Persist results: append, don't overwrite
##### `llm_suggestions.json` is rewritten with all 60 entries (36 production + 24 frontier). `llm_metrics.csv` is appended to — production-tier history stays intact.

In [ ]:
# Save updated suggestions (now contains all 10 LLMs)
suggestions_path = RESULTS_DIR / "llm_suggestions.json"
with open(suggestions_path, "w") as f:
    json.dump(all_suggestions, f, indent=2)
print(f"💾 Saved updated suggestions to {suggestions_path}")

# Append new metrics to the existing CSV
metrics_path = RESULTS_DIR / "llm_metrics.csv"
df_new = pd.DataFrame(new_metrics)

if metrics_path.exists():
    df_existing = pd.read_csv(metrics_path)
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df_combined = df_new

df_combined.to_csv(metrics_path, index=False)
print(f"llm_metrics.csv now has {len(df_combined)} rows ({len(df_new)} new)")

### Validity rates for the 4 new LLMs
##### Apply each frontier LLM's formulas to the real data and record validity.

In [ ]:
new_validity_rows = []
for ds_name, (X, _) in datasets.items():
    print(f"\n{'='*70}")
    print(f"{ds_name.upper()}")
    print("=" * 70)

    for llm_key in FRONTIER_LLMS:
        for variant in prompt_variants:
            key = f"{llm_key}__{variant}"
            features = all_suggestions[ds_name].get(key, [])
            if not features:
                new_validity_rows.append(
                    {
                        "dataset": ds_name,
                        "llm": llm_key,
                        "variant": variant,
                        "n_suggested": 0,
                        "n_applied": 0,
                        "validity_rate": 0.0,
                    }
                )
                print(
                    f"  {llm_key:16s} / {variant:12s}: SKIPPED (no suggestions)"
                )
                continue

            _, report = apply_features(
                X,
                features,
                dataset_name=ds_name,
                llm=llm_key,
                prompt_variant=variant,
                verbose=False,
            )
            new_validity_rows.append(
                {
                    "dataset": ds_name,
                    "llm": llm_key,
                    "variant": variant,
                    "n_suggested": report.n_suggested,
                    "n_applied": report.n_applied,
                    "validity_rate": report.validity_rate,
                }
            )
            print(f"  {report.summary()}")

# Append to existing validity_rates.csv
df_new_validity = pd.DataFrame(new_validity_rows)
validity_path = RESULTS_DIR / "validity_rates.csv"
if validity_path.exists():
    df_existing = pd.read_csv(validity_path)
    df_combined = pd.concat([df_existing, df_new_validity], ignore_index=True)
else:
    df_combined = df_new_validity
df_combined.to_csv(validity_path, index=False)

df_new_validity

### Defensive dedupe of validity_rates.csv
##### If this notebook is re-run, the validity rows for the 4 frontier LLMs would get duplicated in the CSV. Drop duplicates per (dataset, llm, variant), keeping only the latest entry.

In [ ]:
df_validity = pd.read_csv(RESULTS_DIR / "validity_rates.csv")
df_validity = df_validity.drop_duplicates(
subset=["dataset", "llm", "variant"], keep="last"
).reset_index(drop=True)
df_validity.to_csv(RESULTS_DIR / "validity_rates.csv", index=False)
print(f"validity_rates.csv now has {len(df_validity)} unique rows")

### Cost-per-valid-feature: all 10 LLMs together
##### Recompute the efficiency metric across all 10 LLMs (production + frontier combined). The `tier` column distinguishes them so downstream charts can color-code. Sorted ascending — best value at the top.

In [ ]:
df_metrics = pd.read_csv(RESULTS_DIR / "llm_metrics.csv")
df_validity = pd.read_csv(RESULTS_DIR / "validity_rates.csv")

merged = df_metrics.merge(
    df_validity[["dataset", "llm", "variant", "n_applied"]],
    on=["dataset", "llm", "variant"],
    how="left",
)

cpv = (
    merged.groupby("llm")
    .agg(
        total_cost=("cost_usd", "sum"),
        total_valid=("n_applied", "sum"),
        total_suggested=("n_features", "sum"),
        mean_latency=("latency_s", "mean"),
    )
    .reset_index()
)
cpv["cost_per_valid_feature"] = cpv["total_cost"] / cpv["total_valid"].clip(
    lower=1
)
cpv["validity_rate"] = cpv["total_valid"] / cpv["total_suggested"].clip(lower=1)
cpv["tier"] = cpv["llm"].apply(
    lambda x: "frontier" if x in FRONTIER_LLMS else "production"
)
cpv = cpv.sort_values("cost_per_valid_feature")
print(cpv.round(6))

### Stylistic comparison: GPT-5.5 vs DeepSeek V4 on Housing
##### Side-by-side sample of how the two frontier models phrase their formulas. GPT-5.5 wraps every column in `df['...']`; DeepSeek V4 uses bare references. Same correctness, different style — the cause of GPT-5.5's initial 0% Housing validity before the evaluator was patched.

In [ ]:
print("="*70)
print("GPT-5.5 zero-shot — Housing (uses df['col'] style)")
print("="*70)

for f in all_suggestions["housing"].get("gpt-5.5__zero_shot", []):
    print(f"\n• {f.get('name', '?')}")
    print(f"  formula:  {f.get('formula', '?')}")
    print(f"  why:      {f.get('rationale', '?')}")

print("\n" + "="*70)
print("DeepSeek V4 zero-shot — Housing (uses bare column references)")
print("="*70)

for f in all_suggestions["housing"].get("deepseek-v4__zero_shot", [])[:3]:
    print(f"\n• {f.get('name', '?')}")
    print(f"  formula:  {f.get('formula', '?')}")